# LifeBuddy: 01. Exploratory Data Analysis & Behavioral Dynamics

This notebook analyzes the 90-day multi-persona synthetic datasets generated for LifeBuddy.
It investigates the behavioral correlations between sleep, deep focus, daily mood, habit streaks, and personal spending.


In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Connect to database
DB_PATH = os.path.join("..", "lifebuddy.db")
conn = sqlite3.connect(DB_PATH)

print("Connected to SQLite database:", DB_PATH)


## 1. Persona Demographics & Overall Activity Volume


In [2]:
users_df = pd.read_sql_query("SELECT id, username, email, created_at FROM users", conn)
sessions_df = pd.read_sql_query("SELECT * FROM focus_sessions", conn)
tasks_df = pd.read_sql_query("SELECT * FROM tasks", conn)
habits_df = pd.read_sql_query("SELECT * FROM habits", conn)
expenses_df = pd.read_sql_query("SELECT * FROM expenses", conn)
daily_df = pd.read_sql_query("SELECT * FROM daily_logs", conn)

print("--- LifeBuddy Data Summary ---")
print(f"Total Users: {len(users_df)}")
print(f"Total Focus Sessions: {len(sessions_df)}")
print(f"Total Tasks Logged: {len(tasks_df)}")
print(f"Total Habits Tracked: {len(habits_df)}")
print(f"Total Financial Transactions: {len(expenses_df)}")
print(f"Total Daily Wellness Logs: {len(daily_df)}")

users_df


## 2. Circadian Focus Rhythm & Session Durations


In [3]:
# Extract hour of day from completed_at
sessions_df['completed_at'] = pd.to_datetime(sessions_df['completed_at'])
sessions_df['hour'] = sessions_df['completed_at'].dt.hour
sessions_df['weekday'] = sessions_df['completed_at'].dt.day_name()

hour_counts = sessions_df.groupby('hour')['duration_minutes'].sum()
print("Top 5 Focus Hours (by total minutes):")
print(hour_counts.sort_values(ascending=False).head(5))

print("\nSession duration summary (minutes):")
print(sessions_df['duration_minutes'].describe())


## 3. Daily Wellness Dynamics: Sleep vs Focus


In [4]:
# Merge daily focus with daily sleep
daily_focus = sessions_df[sessions_df['mode'] == 'focus'].groupby([
    sessions_df['user_id'], sessions_df['completed_at'].dt.date.astype(str)
])['duration_minutes'].sum().reset_index()
daily_focus.columns = ['user_id', 'date', 'focus_minutes']

wellness_focus = pd.merge(daily_df, daily_focus, on=['user_id', 'date'], how='inner')
print("Sample aligned records (Sleep, Mood, Focus Minutes):")
print(wellness_focus[['user_id', 'date', 'sleep_hours', 'mood', 'focus_minutes']].head(10))

corr = wellness_focus[['sleep_hours', 'mood', 'focus_minutes']].corr()
print("\nPearson Correlation Matrix:")
print(corr)


## 4. Spending Patterns vs Mood


In [5]:
expenses_df['date'] = expenses_df['date'].astype(str)
daily_spending = expenses_df[expenses_df['type'] == 'expense'].groupby(['user_id', 'date'])['amount'].sum().reset_index()

mood_spend = pd.merge(daily_df, daily_spending, on=['user_id', 'date'], how='inner')
print("Spending grouped by Daily Mood rating (1=Low, 5=High):")
print(mood_spend.groupby('mood')['amount'].agg(['count', 'mean', 'median', 'std']))
